# PySpark Retail Data Processing - Spark 3 Version

This notebook has been migrated to Spark 3.x with the following improvements:
- Spark 3 compatibility configurations
- Legacy settings for smooth migration
- Enhanced Adaptive Query Execution (AQE)
- Arrow optimization enabled
- Kryo serialization for better performance

## Migration Notes
- All APIs are compatible with Spark 3.x
- Legacy compatibility settings ensure consistent behavior with Spark 2.x
- No deprecated APIs used


## Spark 3 Configuration

In [ ]:
# Configure Spark 3 settings for optimal performance and compatibility
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Legacy compatibility settings for smooth migration
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.legacy.allowNegativeScaleOfDecimal", "true")
spark.conf.set("spark.sql.storeAssignmentPolicy", "LEGACY")

# Performance optimizations
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print(f"Spark version: {spark.version}")
print("Spark 3 configurations applied successfully!")

## Import Libraries

In [ ]:
from pyspark.sql.functions import col, to_date, year, month, round,sum, desc,row_number
from pyspark.sql import Window

## Load Data

In [ ]:
df = spark.read.csv("/mnt/raw/merge_csv.csv", header=True, inferSchema=True)

In [ ]:
df.count()

## Data Cleaning

In [ ]:
df = df.withColumnRenamed('Customer ID', 'CustomerID')

In [ ]:
df_filterd = df.filter(~col('Invoice').startswith('C'))

In [ ]:
df_filterd = df_filterd.filter(col('CustomerID').isNotNull())

In [ ]:
df_filterd = df_filterd.filter((col('Quantity') > 0) & (col('Price') > 0))

In [ ]:
df_filterd = df_filterd.dropDuplicates()

## Data Transformation

In [ ]:
df_filterd = df_filterd.withColumn('InvoiceDate', to_date(col('InvoiceDate'), 'MM/d/yyyy H:mm'))

df_filterd = df_filterd.withColumn("Year", year(col("InvoiceDate"))) \
    .withColumn("Month", month(col("InvoiceDate")))

In [ ]:
df_filterd = df_filterd.withColumn('Revenue', round(col('Quantity') * col('Price'),2))

## Aggregation

In [ ]:
monthly_sales = df_filterd.groupBy('Year','Month','Stockcode','Description').agg(sum('Revenue').alias('TotalRevenue'))

In [ ]:
display(monthly_sales)

## Product Ranking with Window Functions

In [ ]:
window_spec = Window.partitionBy('Year','Month').orderBy(desc('TotalRevenue'))

ranked = monthly_sales.withColumn('Rank', row_number().over(window_spec))

## Top 10 Products

In [ ]:
top10_product = ranked.filter(col('Rank') <= 10)

In [ ]:
display(top10_product)

## Country-wise Sales Analysis

In [ ]:
country_sales = df_filterd.groupBy('Year','Month','Country').agg(sum('Revenue').alias('TotalRevenue')).orderBy(desc('TotalRevenue'))

In [ ]:
display(country_sales)

## Save Results to Delta Lake

In [ ]:
top10_product.write.format('delta').mode('overwrite').partitionBy('Year','Month').save('/mnt/processed/top10_product')